# TONTOUMA-BOT — Benchmark complet du pipeline (STT → Retrieval → LLM → RAGAS → Système)

Ce notebook évalue **trois modèles Whisper** (tiny, small, fine-tuné Wolof) branchés sur le **même pipeline RAG** (même retriever, même index vectoriel, même LLM), afin d'isoler précisément l'impact du choix du modèle STT sur la qualité globale du système.

Architecture évaluée :

```
Audio Wolof → Whisper (tiny / small / fine-tuné) → Transcription
           → Retrieval (embeddings + recherche) → Contexte récupéré
           → LLM → Réponse finale
           → Métriques STT + Retrieval + LLM/RAGAS + Système
```

**Principe scientifique** : même dataset, même retriever, même LLM, même prompt, même paramètres — seul le modèle STT change entre les trois runs. C'est cette condition qui rend la comparaison valide.

### Prérequis (à exécuter une seule fois)

```bash
pip install jiwer "ragas==0.2.15" psutil datasets langchain-huggingface --break-system-packages
```

### Fichiers attendus dans ce dossier

```
data/
├── orientation_rag.pdf
├── donnees_fictives_procedures_rag.pdf
dataset_test/
├── audio/
│   ├── audio_001.wav
│   ├── audio_002.wav
│   └── ...
└── metadata_test.csv
```

Le fichier `metadata_test.csv` doit contenir au minimum les colonnes : `sample_id, audio_path, reference_transcription, reference_answer, relevant_document_ids` (identifiants séparés par des virgules, ex. `SRV-005,PROC-002`).


## Cellule 1 — Configuration

In [ ]:
import io
import json
import os
import re
import time
import unicodedata
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

warnings.filterwarnings("ignore")

# --- Modèles STT à comparer ---
MODELES_STT = {
    "whisper-tiny":     "openai/whisper-tiny",
    "whisper-small":    "openai/whisper-small",
    "whisper-finetune": "M9and2M/whisper-small-wolof",  # remplacer par ton propre checkpoint fine-tuné si besoin
}

SAMPLE_RATE = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_INDEX = 0 if DEVICE == "cuda" else -1

# --- RAG : mêmes réglages que le notebook TONTOUMA-BOT RAG ---
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TOP_K = 3

DATA_DIR = Path("data")
TEST_DIR = Path("dataset_test")
RESULTS_DIR = Path("resultats_benchmark")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Device utilisé : {DEVICE.upper()}")
print(f"Modèles STT à comparer : {list(MODELES_STT.keys())}")


## Cellule 2 — Chargement des modèles (une seule fois)

In [ ]:
# --- Modèles STT (Whisper) ---
print("Chargement des modèles STT...")
pipelines_stt = {}
tailles_modeles = {}

for nom, chemin in MODELES_STT.items():
    t0 = time.perf_counter()
    pipelines_stt[nom] = pipeline(
        "automatic-speech-recognition",
        model=chemin,
        device=DEVICE_INDEX,
    )
    temps_chargement = time.perf_counter() - t0
    n_params = sum(p.numel() for p in pipelines_stt[nom].model.parameters())
    tailles_modeles[nom] = {
        "n_parametres": n_params,
        "taille_mb": round(n_params * 4 / 1024**2, 1),  # approx float32
        "temps_chargement_s": round(temps_chargement, 2),
    }
    print(f"  {nom:20s} chargé en {temps_chargement:.1f}s — {n_params/1e6:.0f}M paramètres")

# --- Modèle d'embedding pour le RAG ---
print("\nChargement du modèle d'embedding...")
embedder = SentenceTransformer(EMBEDDING_MODEL)

# --- LLM pour la génération de réponse ---
print("Chargement du LLM...")
generator = pipeline("text-generation", model=LLM_MODEL)

print("\nTous les modèles sont chargés.")


## Cellule 3 — Reconstruction du pipeline RAG (base documentaire)

Même logique que le notebook `TONTOUMA_BOT_RAG` : extraction des PDF, découpage en chunks, encodage en embeddings.

In [ ]:
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))
assert len(pdf_paths) == 2, f"2 fichiers PDF attendus dans data/, {len(pdf_paths)} trouvés"

rows = []
for path in pdf_paths:
    reader = PdfReader(path)
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        if not lines:
            continue
        title = lines[0][:80]
        body = "\n".join(lines[1:]) if len(lines) > 1 else lines[0]
        # Identifiant de document (ex. SRV-005, PROC-002) extrait du titre si présent
        match = re.search(r"(SRV|PROC)-\d{3}", title)
        doc_id = match.group(0) if match else f"{path.stem}_p{i+1}"
        rows.append({
            "doc_id": doc_id,
            "source": path.name,
            "page": i + 1,
            "title": title,
            "text": body,
        })

pages = pd.DataFrame(rows)
pages["section"] = pages.apply(lambda r: "## " + r["title"] + "\n\n" + r["text"], axis=1)

chunk_embeddings = embedder.encode(pages["section"].tolist(), normalize_embeddings=True)

print(f"{len(pages)} chunks indexés depuis {len(pdf_paths)} PDF")
print(f"Base vectorielle : {chunk_embeddings.shape}")
display(pages[["doc_id", "source", "page", "title"]].head(10))


In [ ]:
ROLE = (
    "Tu es TONTOUMA-BOT, l'assistant virtuel de l'Hôpital Régional de Diourbel.\n"
    "Voici la documentation officielle de l'hôpital :"
)

CONSIGNE = (
    "Réponds en une ou deux phrases, uniquement à partir des informations "
    "de la documentation ci-dessus. Si l'information ne s'y trouve pas, réponds exactement : "
    '"Je ne sais pas, je vous invite à vous rendre à l\'accueil ou à contacter le poste 100."'
)


def build_prompt(context, question):
    return f"{ROLE}\n\n{context}\n\nQuestion d'un usager : {question}\n{CONSIGNE}"


def ask_llm(prompt):
    conversation = generator([{"role": "user", "content": prompt}], max_new_tokens=80, do_sample=False)
    return conversation[0]["generated_text"][-1]["content"]


def search(question, top_k=TOP_K):
    """Retourne les top_k chunks les plus pertinents ET leurs doc_id (pour les métriques de retrieval)."""
    question_embedding = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = chunk_embeddings @ question_embedding
    top_indices = np.argsort(-similarities)[:top_k]
    results = pages.iloc[top_indices].copy()
    results["score"] = similarities[top_indices]
    return results


def answer_question(question, top_k=TOP_K):
    retrieved = search(question, top_k)
    context = "\n\n".join(retrieved["section"])
    prompt = build_prompt(context, question)
    answer = ask_llm(prompt)
    return answer, retrieved


print("Pipeline RAG prêt (search / answer_question).")


## Cellule 4 — Jeu de test annoté

Le jeu de test doit contenir, pour chaque échantillon audio : la transcription de référence, la réponse attendue, et les identifiants des documents pertinents (vérité terrain pour les métriques de retrieval). **Ce jeu ne doit jamais contenir de fichiers utilisés pour le fine-tuning du modèle Wolof.**

In [ ]:
metadata_path = TEST_DIR / "metadata_test.csv"

if metadata_path.exists():
    test_df = pd.read_csv(metadata_path)
else:
    print("⚠️  Aucun metadata_test.csv trouvé — génération d'un jeu d'exemple à compléter avec de vrais audios.")
    TEST_DIR.mkdir(exist_ok=True)
    (TEST_DIR / "audio").mkdir(exist_ok=True)
    exemple = pd.DataFrame([
        {
            "sample_id": "q001",
            "audio_path": "dataset_test/audio/audio_001.wav",
            "reference_transcription": "Fan la service radiologie ne?",
            "question_reference": "Où se trouve le service de radiologie ?",
            "reference_answer": "Le service de radiologie se trouve au sous-sol du Bâtiment A, aile droite.",
            "relevant_document_ids": "SRV-005",
            "speaker_id": "spk01", "gender": "F", "domain": "orientation",
            "noise_level": "silence", "duration_s": 2.4,
        },
        {
            "sample_id": "q002",
            "audio_path": "dataset_test/audio/audio_002.wav",
            "reference_transcription": "Nan laay def ngir am papier bu deug bi?",
            "question_reference": "Quels documents faut-il pour obtenir un acte de décès ?",
            "reference_answer": "Il faut le certificat médical de décès et une pièce d'identité du déclarant.",
            "relevant_document_ids": "PROC-001",
            "speaker_id": "spk02", "gender": "M", "domain": "administratif",
            "noise_level": "bruit", "duration_s": 3.1,
        },
        {
            "sample_id": "q003",
            "audio_path": "dataset_test/audio/audio_003.wav",
            "reference_transcription": "Naka laay def ngir am rendez-vous ak specialiste bi?",
            "question_reference": "Comment prendre rendez-vous avec un spécialiste ?",
            "reference_answer": "La prise de rendez-vous se fait au guichet 3 ou par téléphone au poste 120.",
            "relevant_document_ids": "PROC-006",
            "speaker_id": "spk01", "gender": "F", "domain": "administratif",
            "noise_level": "silence", "duration_s": 3.8,
        },
    ])
    exemple.to_csv(metadata_path, index=False)
    test_df = exemple
    print(f"Squelette créé : {metadata_path} — REMPLACE les audios et vérifie les références avant le vrai run.")

print(f"\n{len(test_df)} échantillons dans le jeu de test")
display(test_df)


## Cellule 5 — Normalisation de texte et métriques STT (WER / CER)

In [ ]:
try:
    from jiwer import wer as jiwer_wer, cer as jiwer_cer
    JIWER_OK = True
except ImportError:
    JIWER_OK = False
    print("⚠️  jiwer non installé -> pip install jiwer. Utilisation d'un WER Levenshtein local en secours.")


def normaliser_texte(texte: str) -> str:
    texte = str(texte).lower().strip()
    texte = unicodedata.normalize("NFKC", texte)
    texte = re.sub(r"[^\w\sàâäéèêëîïôöùûüÿñç]", " ", texte)
    texte = re.sub(r"\s+", " ", texte)
    return texte


def _wer_local(reference: str, hypothese: str) -> float:
    ref_mots = reference.split()
    hyp_mots = hypothese.split()
    n, m = len(ref_mots), len(hyp_mots)
    matrice = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        matrice[i][0] = i
    for j in range(m + 1):
        matrice[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cout = 0 if ref_mots[i - 1] == hyp_mots[j - 1] else 1
            matrice[i][j] = min(
                matrice[i - 1][j] + 1,
                matrice[i][j - 1] + 1,
                matrice[i - 1][j - 1] + cout,
            )
    return round(matrice[n][m] / max(n, 1), 4)


def calculer_wer(reference: str, hypothese: str) -> float:
    reference = normaliser_texte(reference)
    hypothese = normaliser_texte(hypothese)
    if JIWER_OK:
        return round(jiwer_wer(reference, hypothese), 4)
    return _wer_local(reference, hypothese)


def calculer_cer(reference: str, hypothese: str) -> float:
    reference = normaliser_texte(reference)
    hypothese = normaliser_texte(hypothese)
    if JIWER_OK:
        return round(jiwer_cer(reference, hypothese), 4)
    # CER de secours : Levenshtein au niveau caractère
    ref_c, hyp_c = list(reference.replace(" ", "")), list(hypothese.replace(" ", ""))
    n, m = len(ref_c), len(hyp_c)
    matrice = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        matrice[i][0] = i
    for j in range(m + 1):
        matrice[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cout = 0 if ref_c[i - 1] == hyp_c[j - 1] else 1
            matrice[i][j] = min(matrice[i-1][j]+1, matrice[i][j-1]+1, matrice[i-1][j-1]+cout)
    return round(matrice[n][m] / max(n, 1), 4)


print("Fonctions de normalisation et de calcul WER/CER prêtes.")


## Cellule 6 — Évaluation STT (WER, CER, temps, RTF) pour chaque modèle

In [ ]:
resultats_stt = []

for _, ligne in test_df.iterrows():
    audio_path = ligne["audio_path"]
    reference = ligne["reference_transcription"]
    audio_existe = Path(audio_path).exists()

    for nom_modele, modele in pipelines_stt.items():
        if not audio_existe:
            # Pas de fichier audio réel disponible : on garde la ligne en NA pour ne pas fausser le rapport
            resultats_stt.append({
                "sample_id": ligne["sample_id"],
                "modele_stt": nom_modele,
                "audio_path": audio_path,
                "prediction": None,
                "reference_transcription": reference,
                "wer": np.nan,
                "cer": np.nan,
                "stt_latency_ms": np.nan,
                "rtf": np.nan,
                "echec": True,
            })
            continue

        debut = time.perf_counter()
        try:
            sortie = modele(
                audio_path,
                generate_kwargs={"language": "wolof", "task": "transcribe"},
            )
            prediction = sortie["text"].strip()
            echec = False
        except Exception as e:
            prediction = ""
            echec = True
            print(f"  Erreur {nom_modele} sur {audio_path}: {e}")

        temps_ms = (time.perf_counter() - debut) * 1000
        duree_audio_ms = float(ligne.get("duration_s", np.nan)) * 1000

        resultats_stt.append({
            "sample_id": ligne["sample_id"],
            "modele_stt": nom_modele,
            "audio_path": audio_path,
            "prediction": prediction,
            "reference_transcription": reference,
            "wer": calculer_wer(reference, prediction) if not echec else np.nan,
            "cer": calculer_cer(reference, prediction) if not echec else np.nan,
            "stt_latency_ms": round(temps_ms, 2),
            "rtf": round((temps_ms / 1000) / (duree_audio_ms / 1000), 3) if duree_audio_ms else np.nan,
            "echec": echec,
        })

df_stt = pd.DataFrame(resultats_stt)

if df_stt["echec"].all():
    print("⚠️  Aucun audio réel trouvé dans dataset_test/audio/ — dépose tes fichiers .wav puis relance cette cellule.")
else:
    print(f"{(~df_stt['echec']).sum()} transcriptions réussies sur {len(df_stt)}")

display(df_stt)


## Cellule 7 — Pipeline RAG appliqué à chaque transcription (Expérience B)

Pour chaque transcription produite par chaque modèle STT, on interroge exactement le même retriever et le même LLM, avec mesure de latence détaillée par composant.

In [ ]:
def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


def get_vram_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return None


resultats_rag = []

for item in resultats_stt:
    if item["echec"] or not item["prediction"]:
        continue

    question = item["prediction"]

    ram_avant = get_ram_mb()

    # --- Retrieval ---
    t0 = time.perf_counter()
    retrieved = search(question, top_k=TOP_K)
    retrieval_latency_ms = (time.perf_counter() - t0) * 1000
    contextes = retrieved["section"].tolist()
    retrieved_doc_ids = retrieved["doc_id"].tolist()

    # --- Génération LLM ---
    t0 = time.perf_counter()
    prompt = build_prompt("\n\n".join(contextes), question)
    reponse = ask_llm(prompt)
    llm_latency_ms = (time.perf_counter() - t0) * 1000

    ram_apres = get_ram_mb()

    ligne_ref = test_df[test_df["sample_id"] == item["sample_id"]].iloc[0]

    resultats_rag.append({
        "sample_id": item["sample_id"],
        "modele_stt": item["modele_stt"],
        "user_input": question,
        "retrieved_doc_ids": retrieved_doc_ids,
        "retrieved_contexts": contextes,
        "response": reponse,
        "reference_answer": ligne_ref["reference_answer"],
        "relevant_document_ids": [d.strip() for d in str(ligne_ref["relevant_document_ids"]).split(",")],
        "stt_latency_ms": item["stt_latency_ms"],
        "retrieval_latency_ms": round(retrieval_latency_ms, 2),
        "llm_latency_ms": round(llm_latency_ms, 2),
        "total_latency_ms": round(item["stt_latency_ms"] + retrieval_latency_ms + llm_latency_ms, 2),
        "ram_delta_mb": round(ram_apres - ram_avant, 2),
    })

df_rag = pd.DataFrame(resultats_rag)
print(f"{len(df_rag)} réponses générées via le pipeline RAG complet.")
display(df_rag[["sample_id", "modele_stt", "user_input", "response", "total_latency_ms"]])


## Cellule 8 — Métriques de retrieval (Recall@k, Precision@k, MRR, NDCG@k, Hit@k)

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k=TOP_K):
    retrieved = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    if not relevant:
        return 0.0
    return len(retrieved & relevant) / len(relevant)


def precision_at_k(retrieved_ids, relevant_ids, k=TOP_K):
    retrieved = retrieved_ids[:k]
    if not retrieved:
        return 0.0
    relevant = set(relevant_ids)
    hits = sum(1 for d in retrieved if d in relevant)
    return hits / len(retrieved)


def hit_at_k(retrieved_ids, relevant_ids, k=TOP_K):
    return int(bool(set(retrieved_ids[:k]) & set(relevant_ids)))


def mrr(retrieved_ids, relevant_ids):
    relevant = set(relevant_ids)
    for rang, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rang
    return 0.0


def ndcg_at_k(retrieved_ids, relevant_ids, k=TOP_K):
    relevant = set(relevant_ids)
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k]):
        if doc_id in relevant:
            dcg += 1.0 / np.log2(i + 2)  # i démarre à 0 -> rang réel i+1
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


metriques_retrieval = []
for _, row in df_rag.iterrows():
    metriques_retrieval.append({
        "sample_id": row["sample_id"],
        "modele_stt": row["modele_stt"],
        "recall_at_k": round(recall_at_k(row["retrieved_doc_ids"], row["relevant_document_ids"]), 3),
        "precision_at_k": round(precision_at_k(row["retrieved_doc_ids"], row["relevant_document_ids"]), 3),
        "hit_at_k": hit_at_k(row["retrieved_doc_ids"], row["relevant_document_ids"]),
        "mrr": round(mrr(row["retrieved_doc_ids"], row["relevant_document_ids"]), 3),
        "ndcg_at_k": round(ndcg_at_k(row["retrieved_doc_ids"], row["relevant_document_ids"]), 3),
    })

df_retrieval = pd.DataFrame(metriques_retrieval)
display(df_retrieval)


## Cellule 9 — Métriques RAGAS (fidélité, pertinence, précision/rappel du contexte)

RAGAS évalue le système RAG dans son ensemble — pas le modèle STT isolément. Cette cellule est protégée par un `try/except` : si `ragas` n'est pas installé ou si la configuration LLM échoue, le notebook continue avec les autres métriques.

In [ ]:
RAGAS_OK = False
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

    # On réutilise le même LLM local (Qwen) et le même modèle d'embedding pour rester cohérent
    ragas_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=generator))
    ragas_embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL))

    lignes_ragas = [
        {
            "user_input": row["user_input"],
            "retrieved_contexts": row["retrieved_contexts"],
            "response": row["response"],
            "reference": row["reference_answer"],
        }
        for _, row in df_rag.iterrows()
    ]

    dataset_ragas = Dataset.from_list(lignes_ragas)

    resultats_ragas_raw = evaluate(
        dataset_ragas,
        metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
    )

    df_ragas = resultats_ragas_raw.to_pandas()
    df_ragas["sample_id"] = df_rag["sample_id"].values
    df_ragas["modele_stt"] = df_rag["modele_stt"].values
    RAGAS_OK = True
    display(df_ragas)

except Exception as e:
    print(f"⚠️  RAGAS non exécuté ({type(e).__name__}: {e}).")
    print("   -> pip install \"ragas==0.2.15\" datasets langchain-huggingface")
    df_ragas = pd.DataFrame(columns=[
        "sample_id", "modele_stt", "context_precision", "context_recall",
        "faithfulness", "answer_relevancy",
    ])


## Cellule 10 — Fusion des résultats et sauvegarde CSV

In [ ]:
# --- Fichier détaillé (une ligne par échantillon x modèle) ---
df_final = df_rag.merge(df_retrieval, on=["sample_id", "modele_stt"], how="left")
if RAGAS_OK:
    df_final = df_final.merge(
        df_ragas[["sample_id", "modele_stt", "context_precision", "context_recall", "faithfulness", "answer_relevancy"]],
        on=["sample_id", "modele_stt"], how="left",
    )
else:
    for col in ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]:
        df_final[col] = np.nan

# Rattacher WER/CER
df_final = df_final.merge(
    df_stt[["sample_id", "modele_stt", "wer", "cer", "audio_path"]],
    on=["sample_id", "modele_stt"], how="left",
)

colonnes_finales = [
    "sample_id", "modele_stt", "audio_path", "user_input", "reference_answer", "response",
    "wer", "cer", "stt_latency_ms", "retrieval_latency_ms", "llm_latency_ms", "total_latency_ms",
    "retrieved_doc_ids", "relevant_document_ids", "recall_at_k", "precision_at_k", "hit_at_k", "mrr", "ndcg_at_k",
    "context_precision", "context_recall", "faithfulness", "answer_relevancy",
]
df_final = df_final[[c for c in colonnes_finales if c in df_final.columns]]

chemin_detail = RESULTS_DIR / "evaluation_results.csv"
df_final.to_csv(chemin_detail, index=False)
print(f"Résultats détaillés sauvegardés : {chemin_detail}")

# --- Fichier agrégé (une ligne par modèle) ---
agg_cols = {
    "wer": "mean", "cer": "mean",
    "stt_latency_ms": "mean", "retrieval_latency_ms": "mean", "llm_latency_ms": "mean", "total_latency_ms": "mean",
    "recall_at_k": "mean", "precision_at_k": "mean", "mrr": "mean", "ndcg_at_k": "mean",
    "context_precision": "mean", "context_recall": "mean", "faithfulness": "mean", "answer_relevancy": "mean",
}
agg_cols = {k: v for k, v in agg_cols.items() if k in df_final.columns}

df_summary = df_final.groupby("modele_stt").agg(agg_cols).round(4)
df_summary["wer_std"] = df_final.groupby("modele_stt")["wer"].std().round(4)

chemin_summary = RESULTS_DIR / "evaluation_summary.csv"
df_summary.to_csv(chemin_summary)
print(f"Résultats agrégés sauvegardés : {chemin_summary}\n")

display(df_summary)


## Cellule 11 — Résultats par catégorie (STT / Retrieval / RAG-Système)

In [ ]:
print("=== Résultats STT ===")
display(df_summary[[c for c in ["wer", "cer", "stt_latency_ms"] if c in df_summary.columns]])

print("\n=== Résultats Retrieval ===")
display(df_summary[[c for c in ["recall_at_k", "precision_at_k", "mrr", "ndcg_at_k"] if c in df_summary.columns]])

print("\n=== Résultats RAG et Système ===")
display(df_summary[[c for c in ["faithfulness", "answer_relevancy", "total_latency_ms"] if c in df_summary.columns]])

# Deux gagnants distincts plutôt qu'un score unique masquant le détail
if df_summary["wer"].notna().any():
    meilleur_qualite = df_summary["wer"].idxmin()
    meilleur_vitesse = df_summary["total_latency_ms"].idxmin()
    print(f"\n🏆 Meilleur modèle QUALITÉ (WER le plus bas)  : {meilleur_qualite}")
    print(f"🏆 Meilleur modèle VITESSE (latence la plus basse) : {meilleur_vitesse}")

    # Score global optionnel, à ne jamais substituer aux métriques brutes
    wer_norm = (df_summary["wer"] - df_summary["wer"].min()) / (df_summary["wer"].max() - df_summary["wer"].min() + 1e-9)
    lat_norm = (df_summary["total_latency_ms"] - df_summary["total_latency_ms"].min()) / (df_summary["total_latency_ms"].max() - df_summary["total_latency_ms"].min() + 1e-9)
    df_summary["score_global"] = (0.7 * wer_norm + 0.3 * lat_norm).round(3)
    print(f"\nScore global (0.7×WER + 0.3×latence, plus bas = meilleur) :")
    display(df_summary[["score_global"]].sort_values("score_global"))
else:
    print("\n⚠️ Pas de WER disponible (audios manquants) — dépose les fichiers audio réels pour un classement fiable.")


## Cellule 12 — Expérience A vs Expérience B (isoler l'impact réel de Whisper)

**Expérience A** : question texte de référence directement injectée dans le RAG (sans passer par le STT) — mesure la performance propre du système RAG.
**Expérience B** : question transcrite par chaque modèle Whisper — mesure l'impact réel du STT.

La différence entre A et B représente approximativement la perte de performance causée par la reconnaissance vocale.

In [ ]:
resultats_experience_a = []

for _, ligne in test_df.iterrows():
    question_reference = ligne["question_reference"]

    t0 = time.perf_counter()
    retrieved = search(question_reference, top_k=TOP_K)
    retrieval_latency_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    prompt = build_prompt("\n\n".join(retrieved["section"]), question_reference)
    reponse = ask_llm(prompt)
    llm_latency_ms = (time.perf_counter() - t0) * 1000

    relevant_ids = [d.strip() for d in str(ligne["relevant_document_ids"]).split(",")]
    retrieved_ids = retrieved["doc_id"].tolist()

    resultats_experience_a.append({
        "sample_id": ligne["sample_id"],
        "recall_at_k": round(recall_at_k(retrieved_ids, relevant_ids), 3),
        "mrr": round(mrr(retrieved_ids, relevant_ids), 3),
        "total_latency_ms": round(retrieval_latency_ms + llm_latency_ms, 2),
    })

df_exp_a = pd.DataFrame(resultats_experience_a)

print("=== Expérience A : question texte contrôlée (sans STT) ===")
print(f"Recall@{TOP_K} moyen : {df_exp_a['recall_at_k'].mean():.3f}")
print(f"MRR moyen           : {df_exp_a['mrr'].mean():.3f}")
print(f"Latence moyenne      : {df_exp_a['total_latency_ms'].mean():.0f} ms")

print("\n=== Expérience B : question transcrite par Whisper (par modèle) ===")
for nom_modele in MODELES_STT:
    sous_ensemble = df_final[df_final["modele_stt"] == nom_modele]
    if sous_ensemble.empty or sous_ensemble["recall_at_k"].isna().all():
        continue
    print(f"  {nom_modele:20s} Recall@{TOP_K} = {sous_ensemble['recall_at_k'].mean():.3f}   "
          f"écart vs référence = {df_exp_a['recall_at_k'].mean() - sous_ensemble['recall_at_k'].mean():+.3f}")


## Cellule 13 — Graphiques comparatifs

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

modeles = list(MODELES_STT.keys())
couleurs = ["#4C72B0", "#DD8452", "#55A868"]

# 1. WER moyen par modèle
ax = axes[0, 0]
if df_summary["wer"].notna().any():
    ax.bar(df_summary.index, df_summary["wer"], color=couleurs[:len(df_summary)])
ax.set_title("WER moyen par modèle")
ax.set_ylabel("WER (plus bas = mieux)")
ax.tick_params(axis="x", rotation=20)

# 2. CER moyen par modèle
ax = axes[0, 1]
if "cer" in df_summary.columns and df_summary["cer"].notna().any():
    ax.bar(df_summary.index, df_summary["cer"], color=couleurs[:len(df_summary)])
ax.set_title("CER moyen par modèle")
ax.set_ylabel("CER (plus bas = mieux)")
ax.tick_params(axis="x", rotation=20)

# 3. Latence totale par modèle
ax = axes[0, 2]
ax.bar(df_summary.index, df_summary["total_latency_ms"], color=couleurs[:len(df_summary)])
ax.set_title("Latence totale moyenne par modèle")
ax.set_ylabel("ms")
ax.tick_params(axis="x", rotation=20)

# 4. Recall@k par modèle
ax = axes[1, 0]
ax.bar(df_summary.index, df_summary["recall_at_k"], color=couleurs[:len(df_summary)])
ax.set_title(f"Recall@{TOP_K} par modèle")
ax.set_ylabel("Recall")
ax.tick_params(axis="x", rotation=20)

# 5. Faithfulness et Answer Relevancy
ax = axes[1, 1]
if RAGAS_OK:
    x = np.arange(len(df_summary))
    width = 0.35
    ax.bar(x - width/2, df_summary["faithfulness"], width, label="Faithfulness", color="#4C72B0")
    ax.bar(x + width/2, df_summary["answer_relevancy"], width, label="Answer relevancy", color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels(df_summary.index, rotation=20)
    ax.legend()
else:
    ax.text(0.5, 0.5, "RAGAS non exécuté", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Faithfulness / Answer relevancy")

# 6. Nuage de points WER vs latence (modèle idéal = coin inférieur gauche)
ax = axes[1, 2]
if df_summary["wer"].notna().any():
    for i, (nom, row) in enumerate(df_summary.iterrows()):
        ax.scatter(row["total_latency_ms"], row["wer"], s=150, color=couleurs[i % len(couleurs)], label=nom)
    ax.legend()
ax.set_xlabel("Latence totale (ms)")
ax.set_ylabel("WER")
ax.set_title("WER vs Latence (idéal = coin inférieur gauche)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "graphiques_benchmark.png", dpi=150)
plt.show()

print(f"\nGraphiques sauvegardés : {RESULTS_DIR / 'graphiques_benchmark.png'}")


## Cellule 14 — Ressources système (RAM / VRAM / taille des modèles)

In [ ]:
ram_totale = get_ram_mb()
vram_totale = get_vram_mb()

df_ressources = pd.DataFrame([
    {
        "modele": nom,
        "n_parametres_M": round(infos["n_parametres"] / 1e6, 1),
        "taille_estimee_mb": infos["taille_mb"],
        "temps_chargement_s": infos["temps_chargement_s"],
    }
    for nom, infos in tailles_modeles.items()
])

print(f"RAM totale du process actuel : {ram_totale:.0f} MB")
if vram_totale is not None:
    print(f"VRAM max allouée (CUDA) : {vram_totale:.0f} MB")
else:
    print("VRAM : non applicable (pas de GPU CUDA détecté)")

display(df_ressources)
df_ressources.to_csv(RESULTS_DIR / "ressources_systeme.csv", index=False)


---

## Bilan

Ce benchmark isole précisément l'apport du fine-tuning Wolof en gardant **le même retriever, le même index vectoriel et le même LLM** pour les trois modèles STT — seule la brique de transcription varie.

**Points de lecture essentiels :**

- Le modèle avec le **meilleur WER** n'est pas nécessairement celui avec le **meilleur Recall@k** : une erreur de transcription sur un seul mot-clé peut faire manquer le bon document, même avec un WER global faible.
- L'écart entre l'**Expérience A** (question texte contrôlée) et l'**Expérience B** (question transcrite) mesure la perte de performance réellement imputable à la reconnaissance vocale — c'est la métrique la plus honnête pour juger l'apport du fine-tuning.
- Le **score global** combiné (WER + latence) est fourni à titre indicatif uniquement : il ne doit jamais remplacer la lecture des métriques détaillées dans `evaluation_results.csv`.

### Limites de ce run

- Si `dataset_test/audio/` ne contient pas encore de vrais fichiers `.wav`, les métriques STT et RAG restent vides (`NaN`) — le squelette de `metadata_test.csv` est prêt, il ne manque que les enregistrements réels.
- Le jeu de test actuel est très réduit (3 échantillons) ; pour une évaluation scientifiquement solide, vise 30 à 100 fichiers couvrant plusieurs locuteurs, accents, niveaux de bruit et longueurs de phrase, comme recommandé.
- RAGAS nécessite `ragas`, `datasets` et `langchain-huggingface` installés ; en leur absence, le notebook continue avec les métriques STT et retrieval seules.
